# 07 — Customer Risk Scoring

Create the customer-level file used by Power BI.

In [4]:
from pathlib import Path
import sys, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().resolve()
if not (ROOT/"src").exists(): ROOT = ROOT.parent
sys.path.append(str(ROOT))
DATA = ROOT/"data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
assert DATA.exists(), f"Download the dataset and place it at {DATA}"


In [5]:
import joblib
from src.data_cleaning import clean_data, prepare_target
from src.feature_engineering import add_business_features
from src.predict import risk_level


In [9]:

df=add_business_features(prepare_target(clean_data(pd.read_csv(DATA))))
model=joblib.load(ROOT/'models/final_model1.joblib')
X=df.drop(columns=['Churn','customerID']); p=model.predict_proba(X)[:,1]
out=df[['customerID','tenure','Contract','InternetService','PaymentMethod','MonthlyCharges','Churn']].copy(); out['ChurnProbability']=p; out['RiskLevel']=out.ChurnProbability.map(risk_level)
out.to_csv(ROOT/'data/processed/customer_risk_scores.csv',index=False)
display(out.sort_values('ChurnProbability',ascending=False).head(20)); display(out.RiskLevel.value_counts().to_frame('customers'))

,customerID,tenure,Contract,InternetService,PaymentMethod,MonthlyCharges,Churn,ChurnProbability,RiskLevel
1976,9497-QCMMS,1,Month-to-month,Fiber optic,Electronic check,93.55,1,0.953008,High
3380,5178-LMXOP,1,Month-to-month,Fiber optic,Electronic check,95.10,1,0.952823,High
4800,9300-AGZNL,1,Month-to-month,Fiber optic,Electronic check,94.00,1,0.952601,High
3749,4424-TKOPW,2,Month-to-month,Fiber optic,Electronic check,93.85,1,0.951189,High
2208,7216-EWTRS,1,Month-to-month,Fiber optic,Electronic check,100.80,1,0.951141,High
6368,2720-WGKHP,2,Month-to-month,Fiber optic,Electronic check,94.00,1,0.950998,High
5989,5567-WSELE,3,Month-to-month,Fiber optic,Electronic check,94.60,1,0.950092,High
1410,7024-OHCCK,2,Month-to-month,Fiber optic,Electronic check,93.85,1,0.949780,High
3159,5150-ITWWB,3,Month-to-month,Fiber optic,Electronic check,94.85,0,0.948745,High
997,1374-DMZUI,4,Month-to-month,Fiber optic,Electronic check,94.30,1,0.946165,High


,customers
RiskLevel,
Low,2994
Medium,2337
High,1712


Risk bands are an interpretable starting point. Document the final threshold decision based on the model evaluation and business objective.